# (beta) Dynamic Quantization on an LSTM Word Language Model
https://docs.pytorch.org/tutorials/advanced/dynamic_quantization_tutorial.html

Author: James Reed

Edited by: Seth Weidman

Introduction
Quantization involves converting the weights and activations of your model from float to int, which can result in smaller model size and faster inference with only a small hit to accuracy.

In this tutorial, we will apply the easiest form of quantization - dynamic quantization - to an LSTM-based next word-prediction model, closely following the word language model from the PyTorch examples.

In [ ]:
# imports
import os
from io import open     # 텍스트 파일용 모듈
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Define the model
Here we define the LSTM model architecture, following the model from the word language model example.

In [ ]:
class LSTMModel(nn.Module):
    """Container module with an encoder, a recurrent module, and a decoder."""

    def __init__(self, ntoken, ninp, nhid, nlayers, dropout=0.5):
        super(LSTMModel, self).__init__()  # nn.Module의 생성자 호출
        self.drop = nn.Dropout(dropout)  # 드롭아웃 레이어 정의
        self.encoder = nn.Embedding(ntoken, ninp)  # 단어 임베딩 레이어
        self.rnn = nn.LSTM(ninp, nhid, nlayers, dropout=dropout)  # LSTM 레이어 정의
        self.decoder = nn.Linear(nhid, ntoken)  # 출력을 vocabulary 크기로 매핑하는 선형 레이어

        self.init_weights()  # 가중치 초기화 수행

        self.nhid = nhid  # hidden state 차원 수 저장
        self.nlayers = nlayers  # LSTM 계층 수 저장

    def init_weights(self):
        initrange = 0.1  # 초기화 범위 설정
        self.encoder.weight.data.uniform_(-initrange, initrange)  # 임베딩 가중치 초기화
        self.decoder.bias.data.zero_()  # 디코더 편향 초기화
        self.decoder.weight.data.uniform_(-initrange, initrange)  # 디코더 가중치 초기화

    def forward(self, input, hidden):
        emb = self.drop(self.encoder(input))  # 입력을 임베딩 후 드롭아웃 적용
        output, hidden = self.rnn(emb, hidden)  # LSTM 실행 및 hidden state 업데이트
        output = self.drop(output)  # LSTM 출력에 드롭아웃 적용
        decoded = self.decoder(output)  # 디코더를 통해 vocabulary 차원으로 변환
        return decoded, hidden  # 예측 결과와 hidden state 반환

    def init_hidden(self, bsz):
        weight = next(self.parameters())  # 모델의 임의의 파라미터 가져오기 (장치 일치 목적)
        return (weight.new_zeros(self.nlayers, bsz, self.nhid),  # hidden state 초기화
                weight.new_zeros(self.nlayers, bsz, self.nhid))  # cell state 초기화

## 2. Load in the text data
Next, we load the Wikitext-2 dataset into a Corpus, again following the preprocessing from the word language model example.

In [ ]:
class Dictionary(object):  # 단어와 인덱스를 매핑하는 사전 클래스
    def __init__(self):  # 생성자
        self.word2idx = {}  # 단어 → 인덱스 매핑 딕셔너리
        self.idx2word = []  # 인덱스 → 단어 리스트

    def add_word(self, word):  # 새로운 단어 추가
        if word not in self.word2idx:  # 단어가 처음 등장한 경우
            self.idx2word.append(word)  # 인덱스 리스트에 추가
            self.word2idx[word] = len(self.idx2word) - 1  # 인덱스 매핑 저장
        return self.word2idx[word]  # 단어의 인덱스를 반환

    def __len__(self):  # 사전에 포함된 단어 수 반환
        return len(self.idx2word)


class Corpus(object):  # 말뭉치를 처리하는 클래스
    def __init__(self, path):  # 생성자
        self.dictionary = Dictionary()  # 사전 객체 생성
        self.train = self.tokenize(os.path.join(path, 'train.txt'))  # 학습 데이터 토큰화
        self.valid = self.tokenize(os.path.join(path, 'valid.txt'))  # 검증 데이터 토큰화
        self.test = self.tokenize(os.path.join(path, 'test.txt'))  # 테스트 데이터 토큰화

    def tokenize(self, path):  # 텍스트 파일을 토큰화하는 함수
        """Tokenizes a text file."""
        assert os.path.exists(path)  # 경로가 존재하는지 확인

        # Add words to the dictionary
        with open(path, 'r', encoding="utf8") as f:  # 파일 열기
            for line in f:  # 각 줄에 대해
                words = line.split() + ['<eos>']  # 단어 단위로 나누고 문장 끝 토큰 추가
                for word in words:  # 각 단어에 대해
                    self.dictionary.add_word(word)  # 사전에 단어 추가

        # Tokenize file content
        with open(path, 'r', encoding="utf8") as f:  # 파일 다시 열기
            idss = []  # 문장별 인덱스 리스트 모음
            for line in f:  # 각 줄에 대해
                words = line.split() + ['<eos>']  # 단어 나누고 <eos> 추가
                ids = []  # 한 문장의 단어 인덱스 리스트
                for word in words:  # 각 단어에 대해
                    ids.append(self.dictionary.word2idx[word])  # 사전에서 인덱스를 찾아 추가
                idss.append(torch.tensor(ids).type(torch.int64))  # 텐서로 변환하여 추가
            ids = torch.cat(idss)  # 모든 문장을 하나의 텐서로 연결

        return ids  # 최종 텐서 반환

model_data_filepath = 'data/'  # 데이터 파일 경로 설정

corpus = Corpus(model_data_filepath + 'wikitext-2')  # wikitext-2 데이터셋을 이용해 말뭉치 객체 생성

## 3. Load the pretrained model
This is a tutorial on dynamic quantization, a quantization technique that is applied after a model has been trained. Therefore, we’ll simply load some pretrained weights into this model architecture; these weights were obtained by training for five epochs using the default settings in the word language model example.

Before running this tutorial, download the required pre-trained model:

```bash
wget https://s3.amazonaws.com/pytorch-tutorial-assets/word_language_model_quantize.pth
```
Place the downloaded file in the data directory or update the model_data_filepath accordingly.

In [6]:
ntokens = len(corpus.dictionary)

model = LSTMModel(
    ntoken = ntokens,
    ninp = 512,
    nhid = 256,
    nlayers = 5,
)

model.load_state_dict(
    torch.load(
        model_data_filepath + 'word_language_model_quantize.pth',
        map_location=torch.device('cpu'),
        weights_only=True
        )
    )

model.eval()
print(model)

LSTMModel(
  (drop): Dropout(p=0.5, inplace=False)
  (encoder): Embedding(33278, 512)
  (rnn): LSTM(512, 256, num_layers=5, dropout=0.5)
  (decoder): Linear(in_features=256, out_features=33278, bias=True)
)


Now let’s generate some text to ensure that the pretrained model is working properly - similarly to before, we follow here

In [ ]:
input_ = torch.randint(ntokens, (1, 1), dtype=torch.long)  # 무작위 시작 단어 인덱스 생성 (1x1 크기 텐서)
hidden = model.init_hidden(1)  # 배치 사이즈 1에 맞춰 초기 hidden state 생성
temperature = 1.0  # 샘플링의 randomness 조절을 위한 temperature 설정
num_words = 1000  # 생성할 단어 수

with open(model_data_filepath + 'out.txt', 'w') as outf:  # 결과를 저장할 텍스트 파일 열기
    with torch.no_grad():  # 추론 과정에서는 autograd 비활성화
        for i in range(num_words):  # 설정된 단어 수만큼 반복
            output, hidden = model(input_, hidden)  # 모델에 현재 입력과 hidden state 전달하여 다음 출력 생성
            word_weights = output.squeeze().div(temperature).exp().cpu()  # temperature로 조정된 softmax 확률 분포 생성
            word_idx = torch.multinomial(word_weights, 1)[0]  # 확률 분포에서 단어 인덱스를 샘플링
            input_.fill_(word_idx)  # 다음 입력 단어를 현재 생성된 단어로 설정

            word = corpus.dictionary.idx2word[word_idx]  # 인덱스를 단어로 변환

            outf.write(str(word.encode('utf-8')) + ('\n' if i % 20 == 19 else ' '))  # 20단어마다 줄바꿈하여 파일에 저장

            if i % 100 == 0:  # 100단어마다 진행상황 출력
                print('| Generated {}/{} words'.format(i, 1000))

with open(model_data_filepath + 'out.txt', 'r') as outf:  # 결과 파일 다시 열기
    all_output = outf.read()  # 전체 내용 읽기
    print(all_output)  # 콘솔에 출력

| Generated 0/1000 words
| Generated 100/1000 words
| Generated 200/1000 words
| Generated 300/1000 words
| Generated 400/1000 words
| Generated 500/1000 words
| Generated 600/1000 words
| Generated 700/1000 words
| Generated 800/1000 words
| Generated 900/1000 words
b'losses' b'for' b'than' b'five' b'months' b'more' b'of' b'the' b'years' b'for' b'one' b'TV' b'@-@' b'body' b'specification' b',' b'assistant' b'64' b'hours' b','
b'and' b'Europe' b'levelled' b'from' b'a' b'<unk>' b'positively' b'off' b',' b'reaching' b'a' b'sparrow' b"'s" b'kingdom' b',' b'and' b'especially' b'a' b'more' b'versatile'
b'667' b'team' b'.' b'Again' b',' b'despite' b'stacked' b',' b'geography' b'converting' b'this' b'female' b'and' b'9' b'%' b',' b'and' b'flows' b'its' b'eventual'
b'subplot' b'of' b'seam' b'information' b'.' b'He' b'returned' b'Farm' b'reading' b'for' b'favour' b'with' b'his' b'superiority' b'and' b'interesting' b'skills' b',' b'<unk>' b','
b'and' b'making' b'extreme' b'growth' b'.' b'showcas

It’s no GPT-2, but it looks like the model has started to learn the structure of language!

We’re almost ready to demonstrate dynamic quantization. We just need to define a few more helper functions:

In [ ]:
bptt = 25  # Backpropagation Through Time 길이 설정 (시퀀스 길이)
criterion = nn.CrossEntropyLoss()  # 다중 클래스 분류용 손실 함수
eval_batch_size = 1  # 평가 시 사용할 배치 크기

# create test data set
def batchify(data, bsz):
    # 데이터셋을 bsz로 나누어 배치 생성
    nbatch = data.size(0) // bsz  # 전체 배치를 bsz로 나누었을 때 가능한 배치 수
    data = data.narrow(0, 0, nbatch * bsz)  # 나누어 떨어지지 않는 나머지는 잘라냄
    return data.view(bsz, -1).t().contiguous()  # 배치를 행 단위로 재배열하여 [seq_len, batch_size] 형태로 반환

test_data = batchify(corpus.test, eval_batch_size)  # 테스트 데이터를 배치화

# Evaluation functions
def get_batch(source, i):
    seq_len = min(bptt, len(source) - 1 - i)  # 시퀀스 길이 계산 (끝을 넘지 않도록 제한)
    data = source[i:i+seq_len]  # 입력 시퀀스
    target = source[i+1:i+1+seq_len].reshape(-1)  # 다음 시점의 정답 시퀀스
    return data, target  # 입력과 타깃 반환

def repackage_hidden(h):
    """Wraps hidden states in new Tensors, to detach them from their history."""
    if isinstance(h, torch.Tensor):
        return h.detach()  # Tensor라면 detach해서 이전 연산 기록 제거
    else:
        return tuple(repackage_hidden(v) for v in h)  # tuple이면 재귀적으로 detach

def evaluate(model_, data_source):
    # 평가 모드로 전환 (드롭아웃 비활성화)
    model_.eval()
    total_loss = 0.  # 전체 손실 초기화
    hidden = model_.init_hidden(eval_batch_size)  # 초기 hidden state 설정
    with torch.no_grad():  # 평가 시 그래디언트 계산 비활성화
        for i in range(0, data_source.size(0) - 1, bptt):  # 전체 데이터셋을 bptt 단위로 순회
            data, targets = get_batch(data_source, i)  # 입력 시퀀스와 타깃 획득
            output, hidden = model_(data, hidden)  # 모델 실행
            hidden = repackage_hidden(hidden)  # hidden state detach
            output_flat = output.view(-1, ntokens)  # 출력 텐서를 [전체 토큰 수, 클래스 수] 형태로 펼침
            total_loss += len(data) * criterion(output_flat, targets).item()  # 시퀀스 길이만큼 손실 누적
    return total_loss / (len(data_source) - 1)  # 평균 손실 반환

## 4. Test dynamic quantization
Finally, we can call torch.quantization.quantize_dynamic on the model! Specifically,

We specify that we want the nn.LSTM and nn.Linear modules in our model to be quantized

We specify that we want weights to be converted to int8 values

In [ ]:
import torch.quantization

quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.LSTM, nn.Linear}, dtype=torch.qint8
)
print(quantized_model)

LSTMModel(
  (drop): Dropout(p=0.5, inplace=False)
  (encoder): Embedding(33278, 512)
  (rnn): DynamicQuantizedLSTM(512, 256, num_layers=5, dropout=0.5)
  (decoder): DynamicQuantizedLinear(in_features=256, out_features=33278, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
)


The model looks the same; how has this benefited us? First, we see a significant reduction in model size:

In [ ]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/1e6)
    os.remove('temp.p')

print_size_of_model(model)
print_size_of_model(quantized_model)

Size (MB): 113.944064
Size (MB): 79.738484


Second, we see faster inference time, with no difference in evaluation loss:

Note: we set the number of threads to one for single threaded comparison, since quantized models run single threaded.

In [ ]:
torch.set_num_threads(1)

def time_model_evaluation(model, test_data):
    s = time.time()
    loss = evaluate(model, test_data)
    elapsed = time.time() - s
    print('''loss: {0:.3f}\nelapsed time (seconds): {1:.1f}'''.format(loss, elapsed))

time_model_evaluation(model, test_data)
time_model_evaluation(quantized_model, test_data)

loss: 5.167
elapsed time (seconds): 188.7
loss: 5.168
elapsed time (seconds): 94.7


Running this locally on a MacBook Pro, without quantization, inference takes about 200 seconds, and with quantization it takes just about 100 seconds.

# Conclusion
Dynamic quantization can be an easy way to reduce model size while only having a limited effect on accuracy.

Thanks for reading! As always, we welcome any feedback, so please create an issue here if you have any.

Total running time of the script: ( 5 minutes 23.996 seconds)